# Логирование и raise: контракт preprocessing


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


**Центральная идея:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в ../../data")

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## 1. Схема обязательных столбцов

Напишите require_columns с информативным KeyError.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def require_columns(frame,required):
    # TODO
    ...
assert require_columns(orders,{"order_id","customer_id"}) is True
try: require_columns(orders,{"missing_column"})
except KeyError as e: missing_message=str(e)
assert "missing_column" in missing_message


## 2. Уникальный ключ заказов

Напишите проверку дублей order_id.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def require_unique(frame,column):
    # TODO
    ...
assert require_unique(orders,"order_id") is True
bad=orders.head(2).copy(); bad.loc[bad.index[1],"order_id"]=bad.iloc[0]["order_id"]
try: require_unique(bad,"order_id")
except ValueError as e: duplicate_message=str(e)
assert "order_id" in duplicate_message


## 3. Неотрицательные оплаты

Остановите NaN и отрицательные значения.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def validate_payment_values(frame):
    # TODO
    ...
assert validate_payment_values(payments) is True
bad=payments.head(2).copy(); bad.loc[bad.index[0],"payment_value"]=-1
try: validate_payment_values(bad)
except ValueError as e: payment_message=str(e)
assert "payment_value" in payment_message


## 4. Обязательная дата покупки

Остановите NaT до расчёта Recency.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def validate_purchase_dates(frame):
    # TODO
    ...
assert validate_purchase_dates(orders) is True
bad=orders.head(2).copy(); bad.loc[bad.index[0],"order_purchase_timestamp"]=pd.NaT
try: validate_purchase_dates(bad)
except ValueError as e: date_message=str(e)
assert "timestamp" in date_message


## 5. Единый валидатор

Соберите validate_inputs для трёх таблиц.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def validate_inputs(orders_df,customers_df,payments_df):
    # TODO
    ...
assert validate_inputs(orders,customers,payments) is True


## 6. Лог успешного пути

Запишите шаг только после успешной проверки.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
log=[]  # TODO
validate_inputs(orders,customers,payments)
# TODO: append
merged=None  # TODO; затем append
assert len(log)==2 and log[0].startswith("validated") and log[1].startswith("merged")


## 7. Проверка плохого пути

Докажите, что после raise шаг merge не логируется.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
bad=payments.head(3).copy(); bad.loc[bad.index[0],"payment_value"]=-5
bad_log=[]; caught=""
try:
    # TODO: validate, append, merge
    ...
except ValueError as e: caught=str(e)
assert caught and bad_log==[]


## 8. Инженерная записка

Объясните отличие assert, raise и лога.

**Зачем:** Валидация останавливает неверные данные до признаков, а лог объясняет порядок успешно выполненных шагов. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
CONTRACT_NOTE=""  # TODO
assert len(CONTRACT_NOTE)>=280
assert all(w in CONTRACT_NOTE.lower() for w in ["assert","raise","лог"])
